# Season-level DL pass-rush aggregates (`player_play`)

Manual notebook for Kevin Zhang (BDB 2027). **One output only:** season × player.

**Output (agent-/git-ignored):** `DerivedData/player_play_passrush_by_season_player.csv`

Do **not** paste raw competition CSV rows into AI chats.

## Filters & grain

| Rule | Detail |
|---|---|
| Roster position | Join `players.csv`; keep `nfl_position` ∈ {`DE`,`OLB`,`DT`,`NT`} |
| Grain | One row per `(season, nfl_id)` |
| Season source | `games.season` (optional `season_type == REG`) |

## Column mapping (BDB 2027 snake_case)

| Output / concept | Source column(s) |
|---|---|
| `n_total_plays` | count of filtered player-play rows |
| `n_pass_plays` | `pass_result` ∈ {`C`,`I`,`S`,`IN`,`R`} |
| `n_pass_rush` | `blitzing == True` |
| `wp_added` | If `team_abbr == games.home_team_abbr` → `home_team_win_probability_added`; else → `visitor_team_win_probility_added` (**note official spelling** `probility`) |
| `epa_added` | Play EPA is offense-centric (`expected_points_added`). Credit `+EPA` when `team_abbr == possession_team`, else `-EPA` |
| `n_sacks` | sum of `sack` (0 / 0.5 / 1) |
| `n_pressures` | count non-null `time_to_pressure` |
| `n_quick_pressures` | count where `quick_pressure` is true/nonzero |
| `n_unblocked_pressures` | count where `unblocked_pressure` is true/nonzero |
| `mean/std/n_time_to_pressure` | over non-null `time_to_pressure` |
| `mean/std/n_player_get_off` | over non-null `player_get_off` |
| Identity | `display_name`, `nfl_position` (players), `team_abbr` (mode in season) |


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Paths — edit if your local layout differs
# ---------------------------------------------------------------------------
REPO_ROOT = Path("../..").resolve()
DATA_DIR = REPO_ROOT / "Data"
DERIVED_DIR = REPO_ROOT / "DerivedData"

PLAYER_PLAY = DATA_DIR / "player_play.csv"
GAMES = DATA_DIR / "games.csv"
PLAYERS = DATA_DIR / "players.csv"

OUT_SEASON = DERIVED_DIR / "player_play_passrush_by_season_player.csv"

ROSTER_KEEP = {"DE", "OLB", "DT", "NT"}
PASS_RESULTS = {"C", "I", "S", "IN", "R"}
REG_ONLY = True  # set False to keep PRE/POST as well

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR exists:", DATA_DIR.is_dir())
for p in (PLAYER_PLAY, GAMES, PLAYERS):
    print(f"  {p.name}: {'OK' if p.is_file() else 'MISSING'} -> {p}")


In [ ]:
def _norm(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def resolve_column(columns, *candidates, required: bool = False) -> str | None:
    cols = list(columns)
    by_cf = {c.casefold(): c for c in cols}
    by_norm = {_norm(c): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        if cand.casefold() in by_cf:
            return by_cf[cand.casefold()]
        if _norm(cand) in by_norm:
            return by_norm[_norm(cand)]
    if required:
        raise KeyError(f"Required column not found. Tried: {candidates}. Have: {cols}")
    return None


def headers_only(path: Path) -> list[str]:
    return list(pd.read_csv(path, nrows=0).columns)


for label, path in [("player_play", PLAYER_PLAY), ("games", GAMES), ("players", PLAYERS)]:
    if not path.is_file():
        print(f"[skip] {label}: file missing")
        continue
    cols = headers_only(path)
    print(f"{label}: {len(cols)} columns")
    print(cols)


In [ ]:
assert PLAYER_PLAY.is_file(), f"Missing {PLAYER_PLAY} — set DATA_DIR"
assert GAMES.is_file(), f"Missing {GAMES}"
assert PLAYERS.is_file(), f"Missing {PLAYERS}"

pp_cols = headers_only(PLAYER_PLAY)
games_cols = headers_only(GAMES)
player_cols = headers_only(PLAYERS)

COL = {
    "game_id": resolve_column(pp_cols, "game_id", "gameId", required=True),
    "play_id": resolve_column(pp_cols, "play_id", "playId", required=True),
    "nfl_id": resolve_column(pp_cols, "nfl_id", "nflId", required=True),
    "team_abbr": resolve_column(pp_cols, "team_abbr", "teamAbbr", required=True),
    "possession_team": resolve_column(pp_cols, "possession_team", "possessionTeam", required=True),
    "pass_result": resolve_column(pp_cols, "pass_result", "passResult", required=True),
    "blitzing": resolve_column(pp_cols, "blitzing", required=True),
    "sack": resolve_column(pp_cols, "sack", required=True),
    "time_to_pressure": resolve_column(pp_cols, "time_to_pressure", "timeToPressureAsPassRusher", required=True),
    "player_get_off": resolve_column(pp_cols, "player_get_off", "getOffTimeAsPassRusher", required=True),
    "quick_pressure": resolve_column(pp_cols, "quick_pressure", "quickPressure", required=True),
    "unblocked_pressure": resolve_column(pp_cols, "unblocked_pressure", "unblockedPressure", required=True),
    # WP deltas — visitor column is officially misspelled "probility" in the competition file
    "home_wpa": resolve_column(
        pp_cols,
        "home_team_win_probability_added",
        "homeTeamWinProbabilityAdded",
        required=True,
    ),
    "visitor_wpa": resolve_column(
        pp_cols,
        "visitor_team_win_probility_added",  # official spelling
        "visitor_team_win_probability_added",  # corrected alias if ever fixed
        "visitorTeamWinProbilityAdded",
        "visitorTeamWinProbabilityAdded",
        required=True,
    ),
    "epa": resolve_column(pp_cols, "expected_points_added", "expectedPointsAdded", required=True),
}

GAMES_GAME_ID = resolve_column(games_cols, "game_id", "gameId", required=True)
GAMES_SEASON = resolve_column(games_cols, "season", required=True)
GAMES_TYPE = resolve_column(games_cols, "season_type", "seasonType", "game_type", "gameType")
GAMES_HOME = resolve_column(games_cols, "home_team_abbr", "homeTeamAbbr", required=True)
GAMES_AWAY = resolve_column(games_cols, "visitor_team_abbr", "visitorTeamAbbr", required=True)

PLAYERS_NFL_ID = resolve_column(player_cols, "nfl_id", "nflId", required=True)
PLAYERS_NAME = resolve_column(player_cols, "display_name", "displayName", required=True)
PLAYERS_POS = resolve_column(player_cols, "nfl_position", "position", "nflPosition", required=True)

print("player_play map:")
for k, v in COL.items():
    print(f"  {k:20s} -> {v}")
print("games:", GAMES_GAME_ID, GAMES_SEASON, GAMES_TYPE, GAMES_HOME, GAMES_AWAY)
print("players:", PLAYERS_NFL_ID, PLAYERS_NAME, PLAYERS_POS)


In [ ]:
usecols = list(dict.fromkeys(COL.values()))
pp = pd.read_csv(PLAYER_PLAY, usecols=usecols, na_values=["NA"], low_memory=False)
print("player_play loaded:", pp.shape)

g_usecols = [GAMES_GAME_ID, GAMES_SEASON, GAMES_HOME, GAMES_AWAY]
if GAMES_TYPE:
    g_usecols.append(GAMES_TYPE)
games = pd.read_csv(GAMES, usecols=g_usecols, na_values=["NA"], low_memory=False)
print("games loaded:", games.shape)

players = pd.read_csv(
    PLAYERS,
    usecols=[PLAYERS_NFL_ID, PLAYERS_NAME, PLAYERS_POS],
    na_values=["NA"],
    low_memory=False,
)
print("players loaded:", players.shape)

# Roster-position filter (DE / OLB / DT / NT)
players[PLAYERS_POS] = players[PLAYERS_POS].astype("string").str.strip().str.upper()
players_keep = players.loc[players[PLAYERS_POS].isin(ROSTER_KEEP)].copy()
print(f"players kept by nfl_position in {sorted(ROSTER_KEEP)}: {len(players_keep)}")

df = pp.merge(players_keep, how="inner", left_on=COL["nfl_id"], right_on=PLAYERS_NFL_ID)
df = df.merge(games, how="left", left_on=COL["game_id"], right_on=GAMES_GAME_ID)
print("joined shape:", df.shape)

if REG_ONLY and GAMES_TYPE and GAMES_TYPE in df.columns:
    st = df[GAMES_TYPE].astype("string").str.upper()
    before = len(df)
    df = df.loc[st.isin(["REG", "REGULAR", "REGULAR_SEASON"])].copy()
    print(f"REG filter: {before} -> {len(df)}")


In [ ]:
def to_num(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(float)
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    mapped = s.replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
    return pd.to_numeric(mapped, errors="coerce")


def is_true(s: pd.Series) -> pd.Series:
    return to_num(s).fillna(0) != 0


# Pass / rush flags
pr = df[COL["pass_result"]].astype("string").str.strip().str.upper()
df["is_pass"] = pr.isin(PASS_RESULTS)
df["is_pass_rush"] = is_true(df[COL["blitzing"]])

# WP for the player's team (home vs visitor)
home_wpa = to_num(df[COL["home_wpa"]])
vis_wpa = to_num(df[COL["visitor_wpa"]])
is_home = df[COL["team_abbr"]].astype("string") == df[GAMES_HOME].astype("string")
is_away = df[COL["team_abbr"]].astype("string") == df[GAMES_AWAY].astype("string")
df["wp_added_play"] = np.where(is_home, home_wpa, np.where(is_away, vis_wpa, np.nan))

# EPA for the player's team (offense-centric EPA → sign by possession)
epa = to_num(df[COL["epa"]])
on_offense = df[COL["team_abbr"]].astype("string") == df[COL["possession_team"]].astype("string")
df["epa_added_play"] = np.where(on_offense, epa, -epa)

# Event / timing fields
df["sack_credit"] = to_num(df[COL["sack"]]).fillna(0.0)
df["time_to_pressure"] = to_num(df[COL["time_to_pressure"]])
df["player_get_off"] = to_num(df[COL["player_get_off"]])
df["is_pressure"] = df["time_to_pressure"].notna()
df["is_quick_pressure"] = is_true(df[COL["quick_pressure"]])
df["is_unblocked_pressure"] = is_true(df[COL["unblocked_pressure"]])

n_unmatched_wp = int((~is_home & ~is_away).sum())
print("WP team match issues (not home/away):", n_unmatched_wp)
print("feature null rates / means (no row dump):")
for c in ["wp_added_play", "epa_added_play", "time_to_pressure", "player_get_off", "sack_credit"]:
    s = df[c]
    print(f"  {c:18s} non-null={int(s.notna().sum())} mean={float(s.mean(skipna=True)) if s.notna().any() else float('nan')}")


In [ ]:
group_keys = [GAMES_SEASON, COL["nfl_id"]]


def agg_season(g: pd.DataFrame) -> pd.Series:
    ttp = g["time_to_pressure"]
    go = g["player_get_off"]
    return pd.Series(
        {
            "display_name": g[PLAYERS_NAME].mode(dropna=True).iloc[0]
            if g[PLAYERS_NAME].notna().any()
            else pd.NA,
            "nfl_position": g[PLAYERS_POS].mode(dropna=True).iloc[0]
            if g[PLAYERS_POS].notna().any()
            else pd.NA,
            "team_abbr": g[COL["team_abbr"]].mode(dropna=True).iloc[0]
            if g[COL["team_abbr"]].notna().any()
            else pd.NA,
            "n_total_plays": int(len(g)),
            "n_pass_plays": int(g["is_pass"].sum()),
            "n_pass_rush": int(g["is_pass_rush"].sum()),
            "wp_added": float(g["wp_added_play"].sum(min_count=1)),
            "epa_added": float(g["epa_added_play"].sum(min_count=1)),
            "n_sacks": float(g["sack_credit"].sum()),
            "n_pressures": int(g["is_pressure"].sum()),
            "n_quick_pressures": int(g["is_quick_pressure"].sum()),
            "n_unblocked_pressures": int(g["is_unblocked_pressure"].sum()),
            "mean_time_to_pressure": float(ttp.mean()) if ttp.notna().any() else np.nan,
            "std_time_to_pressure": float(ttp.std(ddof=1)) if ttp.notna().sum() >= 2 else np.nan,
            "n_time_to_pressure": int(ttp.notna().sum()),
            "mean_player_get_off": float(go.mean()) if go.notna().any() else np.nan,
            "std_player_get_off": float(go.std(ddof=1)) if go.notna().sum() >= 2 else np.nan,
            "n_player_get_off": int(go.notna().sum()),
        }
    )


rows = []
for key, g in df.groupby(group_keys, dropna=False, sort=False):
    if not isinstance(key, tuple):
        key = (key,)
    rec = {"season": key[0], "nfl_id": key[1]}
    rec.update(agg_season(g).to_dict())
    rows.append(rec)
by_season = pd.DataFrame(rows)

ordered = [
    "season",
    "nfl_id",
    "display_name",
    "nfl_position",
    "team_abbr",
    "n_total_plays",
    "n_pass_plays",
    "n_pass_rush",
    "wp_added",
    "epa_added",
    "n_sacks",
    "n_pressures",
    "n_quick_pressures",
    "n_unblocked_pressures",
    "mean_time_to_pressure",
    "std_time_to_pressure",
    "n_time_to_pressure",
    "mean_player_get_off",
    "std_player_get_off",
    "n_player_get_off",
]
by_season = by_season[ordered]

print("by_season shape:", by_season.shape)
print("columns:", list(by_season.columns))
print(by_season[["n_total_plays", "n_pass_plays", "n_pass_rush", "wp_added", "epa_added", "n_pressures"]].describe())


In [ ]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
by_season.to_csv(OUT_SEASON, index=False)
print("Wrote:", OUT_SEASON)
print("Done. Keep DerivedData/ out of git and agent-readable paths.")


## Output definitions (quick reference)

- **`n_total_plays`**: rows for that `(season, nfl_id)` after roster filter (+ optional REG).
- **`n_pass_plays`**: `pass_result` in {C,I,S,IN,R}.
- **`n_pass_rush`**: `blitzing` true.
- **`wp_added`**: sum of the WP-added column for the player's side of the ball (home vs visitor via `games`).
- **`epa_added`**: sum of signed `expected_points_added` (`+` on offense, `−` on defense).
- **`n_sacks`**: sum of `sack` credits (half-sacks allowed).
- **`n_pressures`**: non-null `time_to_pressure` count.
- **`n_quick_pressures` / `n_unblocked_pressures`**: true/nonzero counts on those flags.
- **Timing**: mean/std/n over non-null `time_to_pressure` and `player_get_off` (sample std, `ddof=1`).
